## **Modelagem do Dataset e Treinamento do Modelo de Recomendação** ##

In [ ]:
import pandas as pd
import numpy as np
import ast
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# 1. CARREGAR OS DADOS
df = pd.read_csv('archive/movies_metadata_limpo.csv')
df_key = pd.read_csv('archive/keywords.csv')


✅ Tudo pronto! Idiomas traduzidos e arquivo gerado.


In [ ]:
# Aplicando a tradução na coluna original_language
mapa_idiomas = {
    'en': 'English', 'fr': 'French', 'zh': 'Chinese', 'it': 'Italian', 
    'fa': 'Persian', 'nl': 'Dutch', 'de': 'German', 'cn': 'Cantonese', 
    'ar': 'Arabic', 'es': 'Spanish', 'ru': 'Russian', 'sv': 'Swedish', 
    'ja': 'Japanese', 'ko': 'Korean', 'sr': 'Serbian', 'bn': 'Bengali', 
    'he': 'Hebrew', 'pt': 'Portuguese', 'wo': 'Wolof', 'ro': 'Romanian', 
    'hu': 'Hungarian', 'cy': 'Welsh', 'vi': 'Vietnamese', 'cs': 'Czech', 
    'da': 'Danish', 'no': 'Norwegian', 'nb': 'Norwegian Bokmål', 'pl': 'Polish', 
    'el': 'Greek', 'sh': 'Serbo-Croatian', 'xx': 'No Language', 'mk': 'Macedonian', 
    'bo': 'Tibetan', 'ca': 'Catalan', 'fi': 'Finnish', 'th': 'Thai', 
    'sk': 'Slovak', 'bs': 'Bosnian', 'hi': 'Hindi', 'tr': 'Turkish', 
    'is': 'Icelandic', 'ps': 'Pashto', 'ab': 'Abkhazian', 'eo': 'Esperanto', 
    'ka': 'Georgian', 'mn': 'Mongolian', 'bm': 'Bambara', 'zu': 'Zulu', 
    'uk': 'Ukrainian', 'af': 'Afrikaans', 'la': 'Latin', 'et': 'Estonian', 
    'ku': 'Kurdish', 'lv': 'Latvian', 'ta': 'Tamil', 'sl': 'Slovenian', 
    'tl': 'Tagalog', 'ur': 'Urdu', 'rw': 'Kinyarwanda', 'id': 'Indonesian', 
    'bg': 'Bulgarian', 'mr': 'Marathi', 'lt': 'Lithuanian', 'kk': 'Kazakh', 
    'ms': 'Malay', 'sq': 'Albanian', 'qu': 'Quechua', 'te': 'Telugu', 
    'am': 'Amharic', 'jv': 'Javanese', 'tg': 'Tajik', 'ml': 'Malayalam', 
    'hr': 'Croatian', 'lo': 'Lao', 'ay': 'Aymara', 'kn': 'Kannada', 
    'ne': 'Nepali', 'pa': 'Punjabi', 'gl': 'Galician', 'ky': 'Kyrgyz', 
    'sm': 'Samoan', 'eu': 'Basque', 'hy': 'Armenian', 'iu': 'Inuktitut', 
    'si': 'Sinhala'
}

df['original_language'] = df['original_language'].map(mapa_idiomas).fillna('Unknown')


In [ ]:
# 2. Cruzamento das duas bases (metadata + keywords)
df['id'] = pd.to_numeric(df['id'], errors='coerce')
df_key['id'] = pd.to_numeric(df_key['id'], errors='coerce')
df = df.merge(df_key, on='id', how='left')

In [ ]:
# Função para transformar as colunas de listas de dicionários em strings
def limpar_json_para_texto(coluna):
    try:
        lista = ast.literal_eval(coluna)
        return " ".join([d['name'] for d in lista])
    except:
        return ""

# Preparando as colunas para o motor de busca (NLP)
df['keywords_texto'] = df['keywords'].apply(limpar_json_para_texto)

# Como já limpamos 'genres_list' no CSV, garantimos que seja string antes de aplicar o join
df['generos_texto'] = df['genres_list'].apply(lambda x: " ".join(ast.literal_eval(x)) if isinstance(x, str) else " ".join(x))

# Criando a "Sopa de Palavras Chave"
df['sopa_palavras'] = (df['generos_texto'] + " " + df['keywords_texto'] + " " + df['overview'].fillna("")).str.lower()



In [ ]:
runtime_categories = {
    'Very Short': (0, 40),
    'Short': (0, 40),
    'Short-Medium': (41, 100),
    'Medium-Long': (41, 100),
    'Long': (101, 180),
    'Very Long': (181, np.inf)
}

In [ ]:
# Definindo o peso de cada uma das features
features_weights = {
    'vote_average_norm': 0.2,
    'popularity_norm': 0.2,
}

In [ ]:
# 3. FILTRO DE PERFORMANCE
df_app = df#.sort_values('popularity', ascending=False).head(10000).reset_index(drop=True)

# 4. VARIÁVEIS DISCRETAS (O Peso Invisível)
scaler = MinMaxScaler()
v_discretas = scaler.fit_transform(df_app[['popularity_norm', 'roi_relativo', 'vote_average']].fillna(0))

# Criando um Score Final (Pesos: 30% Popularity, 40% ROI, 30% Nota)
df_app['score_discreto'] = (v_discretas[:,0] * 0.3) + (v_discretas[:,1] * 0.4) + (v_discretas[:,2] * 0.3)

# 5. MODELAGEM (TF-IDF para o Prompt)
tfidf = TfidfVectorizer(stop_words='english', max_features=20000)
matriz_tfidf = tfidf.fit_transform(df_app['sopa_palavras'])

# 6. SALVAR O PKL PARA O APP
modelo_final = {
    'df': df_app,
    'matriz_tfidf': matriz_tfidf,
    'tfidf_vectorizer': tfidf
}

with open('archive/modelo_completo.pkl', 'wb') as f:
    pickle.dump(modelo_final, f)

print("✅ Tudo pronto! Idiomas traduzidos e arquivo gerado.")

In [13]:
df[['original_title','original_language']].loc[df['adult'] == True]

,original_title,original_language
27077,Standoff,en
30105,Diet of Sex,es
37642,Amateur Porn Star Killer 2,en
37643,The Band,en
38222,Dværgen,da
38608,Adulterers,en
40374,Half -Life,en


In [19]:
print(df['original_language'].unique())

pd.set_option('display.max_columns', 100)

['en' 'fr' 'zh' 'it' 'fa' 'nl' 'de' 'cn' 'ar' 'es' 'ru' 'sv' 'ja' 'ko'
 'sr' 'bn' 'he' 'pt' 'wo' 'ro' 'hu' 'cy' 'vi' 'cs' 'da' 'no' 'nb' 'pl'
 'el' 'sh' 'xx' 'mk' 'bo' 'ca' 'fi' 'th' 'sk' 'bs' 'hi' 'tr' 'is' 'ps'
 'ab' 'eo' 'ka' 'mn' 'bm' 'zu' 'uk' 'af' 'la' 'et' 'ku' 'lv' 'ta' 'sl'
 'tl' 'ur' 'rw' 'id' 'bg' 'mr' 'lt' 'kk' 'ms' 'sq' 'qu' 'te' 'am' 'jv'
 'tg' nan 'ml' 'hr' 'lo' 'ay' 'kn' 'ne' 'pa' 'gl' 'ky' 'sm' 'eu' 'hy' 'iu'
 'si']


In [29]:
import ast

# Conjunto (set) para armazenar apenas termos únicos
termos_unicos = set()

for linha in df_key['keywords'].dropna():
    try:
        # Transforma a string do CSV em lista de dicionários Python
        lista_dicts = ast.literal_eval(linha)
        
        # Extrai o nome completo da keyword (ex: 'toy comes to life')
        for d in lista_dicts:
            termos_unicos.add(d['name'].strip().lower())
    except:
        continue

print(f"Total de termos (keywords completas) únicos: {len(termos_unicos)}")

Total de termos (keywords completas) únicos: 19950
